# feature_engineering

Input:
    epochs.fif

Output:
    EEG feature table (CSV)

In [1]:
import os

# === 基础库 ===
import numpy as np
import pandas as pd

# === EEG分析 ===
import mne

# === 可视化 ===
import matplotlib.pyplot as plt

# 忽略一些警告
import warnings
warnings.filterwarnings("ignore")

# 打印MNE版本（方便记录环境）
print("MNE version:", mne.__version__)

c:\Users\15122\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


MNE version: 1.11.0


In [2]:
RAW_DIR = r"E:\EEG_project\data\raw"
FIF_DIR = r"E:\EEG_project\data\fif"
OUT_DIR = r"E:\EEG_project\data\processed"
os.makedirs(FIF_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

subject = "sub01"

# 读取epochs

In [3]:
epochs = mne.read_epochs(os.path.join(OUT_DIR, "sub01_epochs_fixed.fif"), preload=True)
print(epochs)

Reading E:\EEG_project\data\processed\sub01_epochs_fixed.fif ...
    Found the data of interest:
        t =       0.00 ...    2000.00 ms
        0 CTF compensation matrices available
Not setting metadata
286 matching events found
No baseline correction applied
0 projection items activated
<EpochsFIF | 286 events (all good), 0 – 2 s (baseline off), ~139.9 MiB, data loaded,
 '1': 286>


In [4]:
print("Epochs shape:", epochs.get_data().shape)
#epochs,channels,timepoints

Epochs shape: (286, 64, 1001)


In [5]:
#只保留EEG通道
epochs = epochs.copy().pick_types(eeg=True)

print("EEG channels:", len(epochs.ch_names))

#打印所有通道名称
names = epochs.ch_names
step = 15
for i in range(0, len(names), step):
    print(", ".join(names[i:i+step]))

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channels: 63
Fp1, Fp2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7
P8, Fz, Cz, Pz, FC1, FC2, CP1, CP2, FC5, FC6, CP5, CP6, FT9, FT10, TP9
TP10, F1, F2, C1, C2, P1, P2, AF3, AF4, FC3, FC4, CP3, CP4, PO3, PO4
F5, F6, C5, C6, P5, P6, AF7, AF8, FT7, FT8, TP7, TP8, PO7, PO8, Fpz
CPz, POz, Oz


# 计算 PSD（Welch 方法）

In [6]:
psds = epochs.compute_psd(
    method="welch",
    fmin=1,
    fmax=45,
    n_fft=1000
)

Effective window size : 2.000 (s)


In [7]:
psd_data = psds.get_data()
freqs = psds.freqs

print("PSD shape:", psd_data.shape)

PSD shape: (286, 63, 89)


#  计算band power和relative power

In [8]:
bands = {
    "delta": (1,4),
    "theta": (4,8),
    "alpha": (8,12),
    "beta": (13,30),
    "gamma": (30,45)
}

In [9]:
band_power = {}

for band, (fmin, fmax) in bands.items():

    freq_mask = (freqs >= fmin) & (freqs <= fmax)

    band_power[band] = psd_data[:, :, freq_mask].sum(axis=2)

print("Alpha band shape:", band_power["alpha"].shape)

Alpha band shape: (286, 63)


In [10]:
total_power = psd_data.sum(axis=2)

relative_power = {}

for band in bands:
    relative_power[band] = band_power[band] / total_power

In [11]:
#检查形状
print(relative_power["alpha"].shape)

(286, 63)


In [12]:
#Relative Power 合理性检查（正常结果应该接近1）
relative_sum = (
    relative_power["delta"] +
    relative_power["theta"] +
    relative_power["alpha"] +
    relative_power["beta"] +
    relative_power["gamma"]
)

print("Mean relative power sum:", relative_sum.mean())

Mean relative power sum: 1.024389858420289


# log transform

In [13]:
log_relative_power = {}

for band in bands:
    
    log_relative_power[band] = np.log(relative_power[band] + 1e-10)# 1e-10是为了避免log(0)

In [14]:
print("Log alpha shape:", log_relative_power["alpha"].shape)

Log alpha shape: (286, 63)


# 计算channel features(每个通道*每个频段，如Fz_alpha)

本质上只是将前面的多个表重新组合排列成一个新的表，符合机器学习用的表格结构，没有运算；
机器学习中每个epoch等于一个样本，所以不要想着合并

In [15]:
channel_names = epochs.ch_names
bands = ["delta", "theta", "alpha", "beta", "gamma"]

feature_dict = {}

for band in bands:
    
    data = log_relative_power[band]   # shape: (epochs, channels)
    
    for ch_idx, ch_name in enumerate(channel_names):
        
        feature_name = f"{ch_name}_{band}"
        
        feature_dict[feature_name] = data[:, ch_idx]


In [16]:
#转为 DataFrame
feature_df = pd.DataFrame(feature_dict)

print(feature_df.shape)
feature_df.head()

(286, 315)


,Fp1_delta,Fp2_delta,F3_delta,F4_delta,C3_delta,C4_delta,P3_delta,P4_delta,O1_delta,O2_delta,...,FT7_gamma,FT8_gamma,TP7_gamma,TP8_gamma,PO7_gamma,PO8_gamma,Fpz_gamma,CPz_gamma,POz_gamma,Oz_gamma
0,-1.310930,-1.044057,-1.259704,-1.007682,-1.149320,-0.515209,-0.836206,-0.874192,-2.028693,-0.305758,...,-2.536427,-3.995049,-1.877233,-4.441129,-1.246510,-1.910725,-1.478287,-3.551442,-2.551358,-2.455205
1,-0.177707,-0.510573,-0.972117,-0.387270,-0.497661,-0.589908,-0.855177,-0.611738,-0.613392,-1.274785,...,-3.281704,-2.887260,-2.745168,-2.346172,-2.633027,-1.561790,-2.899647,-3.690658,-2.577172,-2.542575
2,-3.312816,-0.896658,-1.671914,-1.148300,-0.796402,-0.600296,-1.080035,-0.462435,-1.182772,-1.238946,...,-1.816084,-2.994977,-2.141660,-2.909084,-1.596015,-1.561330,-1.024687,-2.702572,-1.882142,-1.732245
3,-2.512582,-0.932189,-2.286642,-0.929797,-1.350020,-0.982329,-0.742956,-0.889003,-1.203754,-0.546215,...,-1.745792,-3.551695,-1.402525,-2.941946,-2.029996,-1.601110,-1.087081,-3.488979,-2.218774,-2.304118
4,-2.713897,-1.780116,-1.572438,-1.099905,-0.630580,-0.596021,-1.299483,-0.392364,-0.857162,-1.536077,...,-2.429814,-2.045585,-2.308239,-2.516205,-2.263315,-2.023281,-1.186964,-2.803496,-2.399673,-2.293663


# 计算脑区平均（region power）如frontal_alpha

In [17]:
#常见10-20系统简化脑区
regions = {

"frontal": ["Fp1","Fp2","F3","F4","F7","F8","Fz"],
"central": ["C3","C4","Cz"],
"parietal": ["P3","P4","Pz"],
"temporal": ["T7","T8"],
"occipital": ["O1","O2"]

}

In [18]:
for region, ch_list in regions.items():
    
    valid_channels = [ch for ch in ch_list if ch in channel_names]
    
    for band in bands:
        
        cols = [f"{ch}_{band}" for ch in valid_channels]
        
        feature_df[f"{region}_{band}"] = feature_df[cols].mean(axis=1)


# 计算Band Ratios（如frontal_theta_beta_ratio）

In [19]:
ratio_pairs = [

("theta","beta"),
("alpha","beta"),
("theta","alpha"),
("delta","alpha")

]

In [20]:
for region in regions.keys():
    
    for num, den in ratio_pairs:
        
        num_col = f"{region}_{num}"
        den_col = f"{region}_{den}"
        
        ratio_name = f"{region}_{num}_{den}_ratio"
        
        feature_df[ratio_name] = feature_df[num_col] / feature_df[den_col]


# 查看最终 Feature Table

In [21]:
print(feature_df.shape)

feature_df.head()

(286, 360)


,Fp1_delta,Fp2_delta,F3_delta,F4_delta,C3_delta,C4_delta,P3_delta,P4_delta,O1_delta,O2_delta,...,parietal_theta_alpha_ratio,parietal_delta_alpha_ratio,temporal_theta_beta_ratio,temporal_alpha_beta_ratio,temporal_theta_alpha_ratio,temporal_delta_alpha_ratio,occipital_theta_beta_ratio,occipital_alpha_beta_ratio,occipital_theta_alpha_ratio,occipital_delta_alpha_ratio
0,-1.310930,-1.044057,-1.259704,-1.007682,-1.149320,-0.515209,-0.836206,-0.874192,-2.028693,-0.305758,...,0.641766,0.323491,1.238929,1.371608,0.903268,0.113207,1.043227,1.487109,0.701514,0.429605
1,-0.177707,-0.510573,-0.972117,-0.387270,-0.497661,-0.589908,-0.855177,-0.611738,-0.613392,-1.274785,...,0.734342,0.358642,0.838637,1.144519,0.732742,0.272636,0.617651,0.844062,0.731759,0.549643
2,-3.312816,-0.896658,-1.671914,-1.148300,-0.796402,-0.600296,-1.080035,-0.462435,-1.182772,-1.238946,...,0.922761,0.391086,1.360746,1.488252,0.914325,0.346253,2.019874,2.292912,0.880921,0.471736
3,-2.512582,-0.932189,-2.286642,-0.929797,-1.350020,-0.982329,-0.742956,-0.889003,-1.203754,-0.546215,...,0.764590,0.371535,1.127356,1.120922,1.005740,0.560041,2.028186,2.007456,1.010327,0.352284
4,-2.713897,-1.780116,-1.572438,-1.099905,-0.630580,-0.596021,-1.299483,-0.392364,-0.857162,-1.536077,...,1.031886,0.532809,1.154010,1.486789,0.776176,0.276436,1.843907,1.274732,1.446505,0.727400


# 保存 CSV

In [22]:
output_path = r"E:\EEG_project\data\processed\subject01_features.csv"

feature_df.to_csv(output_path, index=False)

print("Feature table saved to:", output_path)


Feature table saved to: E:\EEG_project\data\processed\subject01_features.csv
